# Material Recommender Agent

*Recommends design elements (colors, textures, decorations) from the material library based on user product design queries.*

**Features:**
- Full material library injected into LLM context (no vector search needed at current scale)
- Structured output via Pydantic
- Multi-turn conversation support
- Post-hoc source traceability (report ID + trend heading) populated automatically
- Cross-category creative associations encouraged (beverages can inspire body care)

In [1]:
# Environment setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## State Definitions

Pydantic models for structured recommendation output and the LangGraph state.

In [2]:
%%writefile ../src/deep_research_from_scratch/state_recommender.py

"""State Definitions and Pydantic Schemas for Material Recommendation.

This defines state objects and structured output schemas for the material
recommendation workflow, including the RecommenderState and Pydantic models
for structured recommendation output.
"""

from langgraph.graph import MessagesState
from pydantic import BaseModel, Field

# ===== STRUCTURED OUTPUT SCHEMAS =====

class ImageReference(BaseModel):
    """A reference image matched to a recommended element via embedding retrieval."""

    local_path: str = Field(description="Absolute local file path to the image.")
    description: str = Field(description="Chinese visual description of the image.")


class ElementRecommendation(BaseModel):
    """A single recommended material element from the library."""

    element_id: str = Field(
        description="The element's unique ID from the material library (must match exactly).",
    )
    element_name: str = Field(
        description="The element's Chinese name as it appears in the library.",
    )
    element_name_en: str = Field(
        description="The element's English name as it appears in the library.",
    )
    dimension: str = Field(
        description="The design dimension: 颜色 / 透明度与质地 / 装饰物.",
    )
    reasoning: str = Field(
        description="1-2 sentence explanation of the conceptual link to the user's query.",
    )
    source_reports: list[str] = Field(
        default_factory=list,
        description="Source report IDs — populated via post-hoc lookup, not by the LLM.",
    )
    source_heading: str = Field(
        default="",
        description="Source trend section heading — populated via post-hoc lookup, not by the LLM.",
    )
    reference_images: list[ImageReference] = Field(
        default_factory=list,
        description="Reference images matched via embedding retrieval — populated by attach_images node.",
    )


class RecommendationResult(BaseModel):
    """Structured result of a material recommendation query."""

    concept_analysis: str = Field(
        description=(
            "2-3 sentence analysis of the user's design concept and the key aesthetic "
            "associations driving the recommendations."
        ),
    )
    colors: list[ElementRecommendation] = Field(
        description="Recommended color elements (颜色 dimension).",
    )
    textures: list[ElementRecommendation] = Field(
        description="Recommended texture/transparency elements (透明度与质地 dimension).",
    )
    decorations: list[ElementRecommendation] = Field(
        description="Recommended decoration elements (装饰物 dimension).",
    )


# ===== STATE DEFINITIONS =====

class RecommenderState(MessagesState):
    """State for the material recommender agent.

    Extends MessagesState to support multi-turn conversations,
    storing the latest structured recommendation result for downstream
    consumption and post-hoc source traceability lookup.
    """

    recommendations: RecommendationResult | None = None


Overwriting ../src/deep_research_from_scratch/state_recommender.py


## Material Recommender Agent

Core implementation: library loader, source enrichment, recommend node, and graph.

In [3]:
%%writefile ../src/deep_research_from_scratch/material_recommender.py

"""Material Recommendation Agent.

This module implements a LangGraph-based recommendation agent that suggests
design elements (colors, textures, decorations) from the material library
based on user product design queries. It supports multi-turn conversations
and provides source traceability for all recommendations via post-hoc lookup.
"""

import json
import os
import urllib.parse  # noqa: F401 — pre-load before urllib3 can shadow it
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_openai import AzureOpenAIEmbeddings
from langgraph.graph import END, START, StateGraph

from deep_research_from_scratch.Helper import GenAIToken
from deep_research_from_scratch.prompts import recommender_system_prompt
from deep_research_from_scratch.state_recommender import (
    ElementRecommendation,
    ImageReference,
    RecommendationResult,
    RecommenderState,
)

load_dotenv()

_DEFAULT_RECOMMENDER_MODEL = "azure_openai:GPT-55-2026-04-24"
_MATERIAL_LIBRARY_DIR = Path(__file__).parent.parent.parent / "material_library"
_REPORTS_DIR = Path(__file__).parent.parent.parent / "reports"
_IMAGE_EMBEDDINGS_CACHE = _MATERIAL_LIBRARY_DIR / "image_embeddings.npz"
_TOP_K_IMAGES = 3


def _normalize_model_id(model_id: str) -> str:
    """Normalize Azure model identifiers to use the expected deployment casing."""
    provider, separator, deployment = model_id.partition(":")
    if not separator:
        return model_id
    return f"{provider}{separator}{deployment.upper()}"


def _build_model(model_id: str, **kwargs):
    """Build an Azure OpenAI model instance."""
    normalized_model_id = _normalize_model_id(model_id)
    deployment = normalized_model_id.split(":")[-1]
    return init_chat_model(
        model=normalized_model_id,
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=deployment,
        api_key=GenAIToken().token(),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        default_headers={
            "project-name": os.getenv("HEADERS_PROJECT_NAME"),
            "userid": os.getenv("HEADERS_USERID"),
        },
        **kwargs,
    )


def _build_embedding_model() -> AzureOpenAIEmbeddings:
    """Build an AzureOpenAIEmbeddings client using TEXT-EMBEDDING-3-SMALL."""
    return AzureOpenAIEmbeddings(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment="TEXT-EMBEDDING-3-SMALL",
        api_version="2024-09-01-preview",
        azure_ad_token_provider=lambda: GenAIToken().token(),
        default_headers={
            "project-name": os.getenv("HEADERS_PROJECT_NAME"),
            "userid": os.getenv("HEADERS_USERID"),
        },
    )


def _load_all_image_metadata(reports_dir: Path | None = None) -> list[dict]:
    """Scan all report directories for images_metadata.json and aggregate records.

    Each returned record has 'local_path', 'description', and 'report_id' keys.
    """
    if reports_dir is None:
        reports_dir = _REPORTS_DIR
    records: list[dict] = []
    for meta_path in sorted(reports_dir.glob("*/images/images_metadata.json")):
        report_id = meta_path.parent.parent.name
        try:
            with open(meta_path, encoding="utf-8") as f:
                entries = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue
        for entry in entries:
            local_path = entry.get("local_path", "")
            description = entry.get("description", "")
            if local_path and description:
                records.append({
                    "local_path": local_path,
                    "description": description,
                    "report_id": report_id,
                })
    return records


def _is_cache_valid(records: list[dict], cache_path: Path) -> bool:
    """Return True if the embedding cache exists and is up-to-date."""
    if not cache_path.exists():
        return False
    cached = np.load(str(cache_path), allow_pickle=True)
    cached_metadata = json.loads(str(cached["metadata"]))
    if len(cached_metadata) != len(records):
        return False
    # Invalidate if any images_metadata.json was modified after the cache
    cache_mtime = cache_path.stat().st_mtime
    for meta_path in (_REPORTS_DIR).glob("*/images/images_metadata.json"):
        if meta_path.stat().st_mtime > cache_mtime:
            return False
    return True


def _build_image_index(
    reports_dir: Path | None = None,
    cache_path: Path | None = None,
) -> tuple[np.ndarray, list[dict]]:
    """Build or load the image embedding index.

    Returns (embeddings_matrix, metadata_list) where embeddings_matrix has
    shape (N, D) and metadata_list[i] corresponds to embeddings_matrix[i].
    """
    if cache_path is None:
        cache_path = _IMAGE_EMBEDDINGS_CACHE
    records = _load_all_image_metadata(reports_dir)
    if _is_cache_valid(records, cache_path):
        cached = np.load(str(cache_path), allow_pickle=True)
        embeddings = cached["embeddings"]
        metadata = json.loads(str(cached["metadata"]))
        return embeddings, metadata
    # Build fresh index
    emb_model = _build_embedding_model()
    descriptions = [r["description"] for r in records]
    vectors = np.array(emb_model.embed_documents(descriptions), dtype=float)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        str(cache_path),
        embeddings=vectors,
        metadata=json.dumps(records, ensure_ascii=False),
    )
    return vectors, records


def _build_element_query(element: ElementRecommendation, element_index: dict[str, dict]) -> str:
    """Construct a retrieval query string from an element's name and visual keywords."""
    raw = element_index.get(element.element_id, {})
    keywords: list[str] = raw.get("visual_keywords", [])
    name = element.element_name
    if keywords:
        return f"{name} {' '.join(keywords)}"
    return name


def _search_images(
    query: str,
    embeddings: np.ndarray,
    metadata: list[dict],
    emb_model: AzureOpenAIEmbeddings,
    top_k: int = _TOP_K_IMAGES,
) -> list[ImageReference]:
    """Retrieve top-k most similar images to a query via cosine similarity."""
    q_vec = np.array(emb_model.embed_query(query), dtype=float)
    # Cosine similarity: dot product of unit vectors
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    normed = embeddings / norms
    q_norm = q_vec / (np.linalg.norm(q_vec) or 1.0)
    scores = normed @ q_norm
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        m = metadata[idx]
        results.append(ImageReference(local_path=m["local_path"], description=m["description"]))
    return results


def load_material_library(library_dir: Path | None = None) -> str:
    """Load and format all material library elements for LLM context injection.

    Source traceability fields are excluded from prompt context and added
    via post-hoc lookup after recommendations are produced.
    """
    if library_dir is None:
        library_dir = _MATERIAL_LIBRARY_DIR

    files = {
        "颜色": library_dir / "color.json",
        "透明度与质地": library_dir / "texture.json",
        "装饰物": library_dir / "decoration.json",
    }

    lines = []
    for dimension_label, filepath in files.items():
        if not filepath.exists():
            raise FileNotFoundError(f"Material library file not found: {filepath}")
        try:
            with open(filepath, encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Malformed JSON in {filepath}: {e}") from e

        elements = data.get("elements", [])
        lines.append(f"\n### {dimension_label} ({len(elements)} 个元素)\n")
        for elem in elements:
            keywords = ",".join(elem.get("visual_keywords", [])[:10])
            signals = ",".join(elem.get("signals", [])[:10])
            use = elem.get("typical_use", "")
            name = elem.get("name", "")
            name_en = elem.get("name_en", "")
            elem_id = elem.get("id", "")
            line = (
                f"[{dimension_label}] {name} / {name_en} (id:{elem_id})"
                f" | keywords: {keywords}"
                f" | signals: {signals}"
                f" | use: {use}"
            )
            lines.append(line)

    return "\n".join(lines)


def _build_element_index(library_dir: Path | None = None) -> dict[str, dict]:
    """Build an index of all material library elements keyed by element ID."""
    if library_dir is None:
        library_dir = _MATERIAL_LIBRARY_DIR

    index: dict[str, dict] = {}
    for filename in ["color.json", "texture.json", "decoration.json"]:
        filepath = library_dir / filename
        if not filepath.exists():
            continue
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
        for elem in data.get("elements", []):
            elem_id = elem.get("id", "")
            if elem_id:
                index[elem_id] = elem
    return index


def _enrich_with_sources(
    result: RecommendationResult,
    element_index: dict[str, dict],
) -> RecommendationResult:
    """Populate source traceability fields via post-hoc element lookup."""

    def enrich_list(recs: list[ElementRecommendation]) -> list[ElementRecommendation]:
        enriched = []
        for rec in recs:
            elem = element_index.get(rec.element_id, {})
            raw_source = elem.get("source_report", "")
            source_reports = list(
                dict.fromkeys(s.strip() for s in raw_source.split(" + ") if s.strip())
            )
            source_heading = elem.get("source_heading", "")
            enriched.append(
                rec.model_copy(update={"source_reports": source_reports, "source_heading": source_heading})
            )
        return enriched

    return RecommendationResult(
        concept_analysis=result.concept_analysis,
        colors=enrich_list(result.colors),
        textures=enrich_list(result.textures),
        decorations=enrich_list(result.decorations),
    )


def _format_recommendations_as_text(result: RecommendationResult) -> str:
    """Format a RecommendationResult as readable markdown for conversation history."""
    lines = [f"**概念分析**: {result.concept_analysis}\n"]
    for dimension_label, recs in [
        ("候选颜色", result.colors),
        ("候选质地", result.textures),
        ("候选装饰物", result.decorations),
    ]:
        lines.append(f"### {dimension_label}")
        for i, rec in enumerate(recs, 1):
            source_info = ""
            if rec.source_heading:
                report_ids = [r.split("/")[0][:8] for r in rec.source_reports]
                source_info = f" *(来源: {rec.source_heading}，报告: {', '.join(report_ids)})*"
            lines.append(
                f"{i}. **{rec.element_name}** ({rec.element_name_en})\n"
                f"   {rec.reasoning}{source_info}"
            )
            for img in rec.reference_images:
                lines.append(f"   📷 {img.description} ({img.local_path})")
        lines.append("")
    return "\n".join(lines)


def recommend(state: RecommenderState, config: RunnableConfig) -> dict:
    """Generate material recommendations based on conversation state.

    Loads the full material library, builds the system prompt, calls LLM with
    structured output, then enriches results with source traceability via post-hoc
    lookup. Multi-turn conversations are supported through full message history.

    Model is controlled by config["configurable"]["recommender_model"]
    (default: "azure_openai:gpt-4.1").
    """
    configurable = config.get("configurable", {})
    model = _build_model(
        configurable.get("recommender_model", _DEFAULT_RECOMMENDER_MODEL),
        temperature=0.7,
    )
    structured_model = model.with_structured_output(RecommendationResult)

    material_library_text = load_material_library()
    system_content = recommender_system_prompt.format(material_library=material_library_text)

    messages = [SystemMessage(content=system_content)] + list(state["messages"])
    result: RecommendationResult = structured_model.invoke(messages)

    element_index = _build_element_index()
    result = _enrich_with_sources(result, element_index)

    return {
        "messages": [],
        "recommendations": result,
    }


def attach_images(state: RecommenderState) -> dict:
    """Attach reference images to each recommended element via embedding retrieval.

    Loads (or builds) the image embedding index, then for each ElementRecommendation
    in the current recommendations, computes a query from the element's name and
    visual keywords and retrieves the top-k most similar images.
    """
    result: RecommendationResult | None = state.get("recommendations")
    if result is None:
        return {}

    element_index = _build_element_index()
    embeddings, metadata = _build_image_index()
    emb_model = _build_embedding_model()

    def attach_to_list(recs: list[ElementRecommendation]) -> list[ElementRecommendation]:
        updated = []
        for rec in recs:
            query = _build_element_query(rec, element_index)
            images = _search_images(query, embeddings, metadata, emb_model)
            updated.append(rec.model_copy(update={"reference_images": images}))
        return updated

    enriched_result = RecommendationResult(
        concept_analysis=result.concept_analysis,
        colors=attach_to_list(result.colors),
        textures=attach_to_list(result.textures),
        decorations=attach_to_list(result.decorations),
    )

    recommendations_text = _format_recommendations_as_text(enriched_result)
    return {
        "messages": [AIMessage(content=recommendations_text)],
        "recommendations": enriched_result,
    }


def _build_graph() -> StateGraph:
    """Construct the material recommender StateGraph."""
    graph = StateGraph(RecommenderState)
    graph.add_node("recommend", recommend)
    graph.add_node("attach_images", attach_images)
    graph.add_edge(START, "recommend")
    graph.add_edge("recommend", "attach_images")
    graph.add_edge("attach_images", END)
    return graph


recommender_agent = _build_graph().compile()


Overwriting ../src/deep_research_from_scratch/material_recommender.py


## Preview: Material Library Format

Inspect what the LLM sees (first 10 lines of context).

In [4]:
from deep_research_from_scratch.material_recommender import load_material_library

library_text = load_material_library()
print(library_text[:2000])


### 颜色 (38 个元素)

[颜色] 低饱和香氛色 / Low-Saturation Fragrance Palette (id:25fccecd-color-90b96051-26da05) | keywords: 花香调,香水感,高级香氛联想,香氛沐浴露,柔和花香感,清新香氛,轻香氛感,高端身体护理,高端乳液,经典沐浴油 | signals: 高端香氛延伸,细致香氛感,香氛身体护理感,香氛型身体护理,高级香水化,奢雅身体护理,系列化香水同线体验,香水同线身体乳体验,轻盈香氛体验,高端身体护理 | use: 高端身体乳、沐浴液、液体皂等香氛身体护理产品
[颜色] 淡彩颗粒色系 / Pastel Granule Palette (id:25fccecd-color-35cf7057-42f926) | keywords: 淡粉,淡黄,淡蓝,低饱和彩粒,清新套装感 | signals: 年轻化,多香型区分,轻甜趣味,社媒友好 | use: 身体磨砂膏、颗粒磨砂套装
[颜色] 趣味联名粉蓝米色 / Playful Pink-Blue-Beige Palette (id:25fccecd-color-095cec6b-9bf8a9) | keywords: 粉色,蓝色,米色,联名配色,年轻跳色,联名感,年轻化,跳色组合,粉蓝配色,低饱和 | signals: 年轻社媒化,趣味联名,情绪价值,多感官吸引,年轻社媒属性,趣味收藏感,多角色/多香型区分,年轻潮流,联名话题性,社交媒体友好 | use: 联名身体磨砂、限定护理产品
[颜色] 白茶香草黄 / White Tea Vanilla Yellow (id:25fccecd-color-bafd4ad5-9054f6) | keywords: 淡黄,暖调,白茶,香草,柑橘,轻甜 | signals: 白茶或柑橘香型联想,温暖疗愈,天然滋养感,阳光愉悦 | use: 白茶、香草、柑橘等暖香型身体磨砂配色
[颜色] 自然香型映射色彩 / Nature-Inspired Scent-Mapped Colors (id:25fccecd-color-9a98e69a-93014b) | keywords: 森林联想,花园联想,无花果联想,自然色彩想象,香型映射,植物氛围 | signals: 香型可视化,自然叙事,场景化体

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage
import uuid

from utils import init_langfuse_tracing
from deep_research_from_scratch.material_recommender import _build_model, load_material_library
from deep_research_from_scratch.prompts import recommender_system_prompt

# Langfuse tracing config (same pattern as notebook 5)
langfuse_handler = init_langfuse_tracing()
thread = {
    "configurable": {
        "thread_id": str(uuid.uuid4()),
        "recommender_model": "azure_openai:GPT-55-2026-04-24",
    },
    "callbacks": [langfuse_handler],
    "metadata": {
        "notebook": "7_material_recommender",
        "langfuse_session_id": "20260520",
    },
}

# Compute how many input tokens are contributed by material_library text in system prompt
model = _build_model(thread["configurable"]["recommender_model"], temperature=0.7)
library_text = load_material_library()
system_with_library = recommender_system_prompt.format(material_library=library_text)
system_without_library = recommender_system_prompt.format(material_library="")

library_in_prompt_tokens = model.get_num_tokens(system_with_library) - model.get_num_tokens(system_without_library)
estimated_input_tokens = None
try:
    probe_messages = [
        SystemMessage(content=system_with_library),
        HumanMessage(content="token probe"),
    ]
    estimated_input_tokens = model.get_num_tokens_from_messages(probe_messages)
except NotImplementedError:
    # Some Azure model versions do not implement message-level token counting in LangChain.
    pass

print(f"library_in_prompt_tokens: {library_in_prompt_tokens}")
if estimated_input_tokens is None:
    print("estimated_input_tokens: unavailable for this model in LangChain (use Langfuse trace for actual usage)")
else:
    print(f"estimated_input_tokens (system+probe): {estimated_input_tokens}")
print("Langfuse callback initialized. Pass config=thread when invoking recommender_agent.")

library_in_prompt_tokens: 12842
estimated_input_tokens: unavailable for this model in LangChain (use Langfuse trace for actual usage)
Langfuse callback initialized. Pass config=thread when invoking recommender_agent.


## Test 1: End-to-End Recommendation

Test the recommender agent with a yogurt-concept shower gel query and verify output format and source traceability.

In [12]:
from langchain_core.messages import HumanMessage
from deep_research_from_scratch.material_recommender import recommender_agent

# End-to-end test: yogurt concept shower gel
result = recommender_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="设计一款以酸奶为设计概念的沐浴露，有什么推荐的颜色/质地/装饰物元素？每个维度推荐5个候选元素"
            )
        ]
    },
    config=thread,
 )

print(result["messages"][-1].content)
print("\n--- Structured Output ---")
recs = result["recommendations"]
print(f"concept_analysis: {recs.concept_analysis[:100]}...")
print(f"\nColors count: {len(recs.colors)}")
print(f"Textures count: {len(recs.textures)}")
print(f"Decorations count: {len(recs.decorations)}")

# Verify source traceability
print("\n--- Source Traceability Check ---")
for rec in recs.decorations[:3]:
    print(f"  {rec.element_name}: source_heading={rec.source_heading!r}, source_reports={rec.source_reports[:1]}")

print("\n✓ End-to-end test complete")
print("Check Langfuse traces for actual token usage on this run.")

**概念分析**: 酸奶概念沐浴露的核心可以围绕“乳白、绵密、轻发酵、果味分层”和“温和亲肤”展开：既要有酸奶般柔软 creamy 的视觉，也要保留沐浴露清爽、易冲洗、起泡的使用联想。设计上可借鉴酸奶杯、果酱酸奶、希腊酸奶和分层乳饮的语言，把乳白基底、果味色带、细腻泡沫和可视化小颗粒结合起来。

### 候选颜色
1. **米色** (Beige)
   米色能传达酸奶、乳霜和温润养护的基础感，适合作为酸奶沐浴露的主色。它比纯白更柔和，能强化亲肤、日常、低刺激的产品气质。 *(来源: 1.4 三国横向比较，报告: e9f1f27f)*
2. **米白功效色** (Off-White Efficacy Tone)
   米白色很适合表现“原味酸奶”的干净乳感，同时带有温和稳定的功效护肤联想。用于沐浴露时，可以让产品看起来更像一款身体屏障护理型清洁产品。 *(来源: ## 9. 结论：近半年中日韩面护料体设计的核心方向，报告: e9f1f27f)*
3. **白色细泡** (White Fine Foam)
   白色细泡能够直接连接酸奶的乳白感与沐浴露的清洁起泡感。它适合用于主视觉或料体展示，强调绵密、洁净、温和不刺激。 *(来源: 绵密泡沫 | 白色细泡、蓬松堆积，报告: e9f1f27f)*
4. **淡彩颗粒色系** (Pastel Granule Palette)
   淡粉、淡黄、淡蓝等低饱和色可以转译为草莓酸奶、蜜桃酸奶、蓝莓酸奶等多香型系列。柔和淡彩既有年轻感，也不会破坏酸奶概念的温和乳感。 *(来源: 淡彩颗粒磨砂与糖晶质地，报告: e602b08d)*
5. **绿色分层色** (Layered Green Tone)
   绿色分层色可用于表现牛油果酸奶、青提酸奶或植物乳酸菌概念，带来清新健康的差异化。它与乳白基底结合时，很容易形成“果蔬酸奶杯”的视觉记忆点。 *(来源: 4. 酸奶/奶昔/冰沙：厚重、发酵、果肉感成为“健康甜品”视觉，报告: f7b105e7)*

### 候选质地
1. **不透明** (Opaque)
   不透明质地最能体现酸奶的乳状厚度和高营养密度感，适合做成乳白型沐浴露或沐浴乳。它能让产品显得更滋润、更有包裹感。 *(来源: 本报告只讨论“内容物本身”，报告: f5b757a1)*
2. **轻盈丝滑易推开肤感** (L

/home/shuya/deep_research_from_scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RecommendationResult(conc...[], source_heading='')]), input_type=RecommendationResult])
  return self.__pydantic_serializer__.to_python(


## Test 2: Multi-Turn Interaction

Test refinement interactions: 换一批 (new batch) and style constraint (更偏清新的).

In [1]:
# Multi-turn test using LangGraph thread-based state
from langchain_core.messages import HumanMessage
from deep_research_from_scratch.material_recommender import recommender_agent

# Turn 1: initial query
state1 = recommender_agent.invoke({
    "messages": [HumanMessage(content="以滋养+温泉为核心概念的沐浴露，每个维度推荐3个装饰物元素")]
})
print("=== Turn 1 ===")
print(state1["messages"][-1].content)

# Turn 2: request different recommendations
state2 = recommender_agent.invoke({
    "messages": state1["messages"] + [
        HumanMessage(content="换一批，给我另外3个装饰物候选")
    ]
})
print("\n=== Turn 2 (换一批) ===")
print(state2["messages"][-1].content)

# Check no repeated element_ids across turns
ids_t1 = {r.element_id for r in state1["recommendations"].decorations}
ids_t2 = {r.element_id for r in state2["recommendations"].decorations}
repeats = ids_t1 & ids_t2
print(f"\n✓ Repeated element_ids (should be 0 or minimal): {len(repeats)} - {repeats}")

print("\n✓ Multi-turn test complete")

/home/shuya/deep_research_from_scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RecommendationResult(conc...[], source_heading='')]), input_type=RecommendationResult])
  return self.__pydantic_serializer__.to_python(


=== Turn 1 ===
**概念分析**: “滋养 + 温泉”沐浴露适合表达为温润、矿物感、SPA 仪式感与沐浴后肌肤柔亮的组合：视觉上可用暖白、米色、低饱和香氛色来建立舒缓高级感。质地建议突出清透水感与油润包裹的平衡，装饰物则可用泡沫、油层、草本/矿物悬浮感强化“温泉护理”的可感知性。

### 候选颜色
1. **低饱和香氛色** (Low-Saturation Fragrance Palette)
   低饱和香氛色能把沐浴露从普通清洁带向高端香氛身体护理语境，适合温泉概念中的放松、疗愈和柔和氛围。用于瓶身或料体色彩时，可营造温润而不刺激的 SPA 感。 *(来源: 低饱和“香氛色”：粉、绿、紫、乳白成为高端身体乳/沐浴液的主色，报告: e602b08d, e9f1f27f, f5b757a1, f7b105e7)*
2. **米色** (Beige)
   米色具有柔和暖调和亲肤感，能很好地承接“滋养”概念，传递温润包裹与日常适用的身体护理体验。它也容易联想到矿物泥、温泉水汽和天然疗愈空间。 *(来源: 1.4 三国横向比较，报告: e9f1f27f)*
3. **白茶香草黄** (White Tea Vanilla Yellow)
   白茶香草黄带有暖调、轻甜和阳光疗愈感，适合作为滋养型沐浴露的柔和香型视觉。它能让温泉概念不显得冷硬，而是更有舒适、放松、沐浴后肌肤被滋润的联想。 *(来源: ### 4. 身体磨砂：颗粒、晶体、双相油层是最强视觉品类，报告: e602b08d)*

### 候选质地
1. **透明液体感** (Transparent Liquid Texture)
   透明液体感非常适合沐浴露表达清爽水感，与“温泉水”概念直接相关，能传递清透、洁净、流动的洗浴体验。若加入微微柔雾或淡色调，还能兼顾滋养而不厚重的感受。 *(来源: 低饱和“香氛色”：粉、绿、紫、乳白成为高端身体乳/沐浴液的主色，报告: e602b08d, e9f1f27f, f5b757a1, f7b105e7)*
2. **盐油 SPA 质地** (Salt-in-Oil Spa Texture)
   盐油 SPA 质地能够把温泉矿物、居家 SPA 与滋润护理连接起来，尤其适合表现“矿物滋养”或“温泉盐感”的产品故事。即使实际产品是沐浴露，也可借鉴其油润包裹、颗

## Test 3: Image Embedding Index

Verify cache build, cache hit, and cache invalidation.

In [ ]:
import shutil
from pathlib import Path
import numpy as np
from deep_research_from_scratch.material_recommender import (
    _load_all_image_metadata,
    _build_image_index,
    _is_cache_valid,
)

# Test: load metadata
records = _load_all_image_metadata()
print(f"Total image records loaded: {len(records)}")
assert len(records) > 0
assert all("local_path" in r and "description" in r and "report_id" in r for r in records)
print("✓ _load_all_image_metadata returns valid records")

# Test: build index (first time)
test_cache = Path("/tmp/test_image_embeddings.npz")
if test_cache.exists():
    test_cache.unlink()

embs, meta = _build_image_index(cache_path=test_cache)
print(f"Built index: embeddings shape={embs.shape}, metadata count={len(meta)}")
assert embs.shape[0] == len(meta)
assert test_cache.exists()
print("✓ First-time index build succeeded")

# Test: cache hit
embs2, meta2 = _build_image_index(cache_path=test_cache)
assert embs2.shape == embs.shape
print("✓ Cache hit: returned same shape embeddings")

# Test: cache invalidation by count mismatch
np.savez(str(test_cache), embeddings=embs[:5], metadata='[]')
assert not _is_cache_valid(records, test_cache)
print("✓ Cache invalidation works for count mismatch")

test_cache.unlink(missing_ok=True)
print("\nAll image index tests passed!")

## Test 4: End-to-End Recommendation with Reference Images

In [ ]:
from langchain_core.messages import HumanMessage
from deep_research_from_scratch.material_recommender import recommender_agent

config = {"configurable": {"thread_id": "test-images-001"}}
query = "设计一款以酸奶为设计概念的沐浴露，推荐颜色/质地/装饰物各3个候选"

result = recommender_agent.invoke(
    {"messages": [HumanMessage(content=query)]},
    config=config,
)

recs = result["recommendations"]
print(f"Concept analysis: {recs.concept_analysis[:100]}...")
print(f"Colors: {len(recs.colors)}, Textures: {len(recs.textures)}, Decorations: {len(recs.decorations)}")

for dim_label, elems in [("颜色", recs.colors), ("质地", recs.textures), ("装饰物", recs.decorations)]:
    for elem in elems:
        print(f"  [{dim_label}] {elem.element_name}: {len(elem.reference_images)} reference images")
        for img in elem.reference_images:
            print(f"    📷 {img.description[:50]} | {img.local_path.split('/')[-1]}")
        assert len(elem.reference_images) > 0, f"Element {elem.element_name} should have reference images"

print("\n=== Final formatted message ===")
print(result['messages'][-1].content[:2000])
print("\n✓ End-to-end test with reference images passed!")